# Day 039 Solution — Chart Dashboard

bar_chart, line_chart, scatter_chart, histogram, and multi_chart_figure. All data defined inline. Saves `chart_dashboard.png`.

In [ ]:
import warnings
warnings.filterwarnings('ignore')
import os
import io
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import pandas as pd

import matplotlib.pyplot as plt

def bar_chart(ax, labels, values, title='', xlabel='', ylabel=''):
    ax.bar(labels, values)
    ax.set_title(title)
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    ax.tick_params(axis='x', rotation=45)
    return ax


import matplotlib.pyplot as plt

def line_chart(ax, x, y, title='', xlabel='', ylabel='', label=None):
    ax.plot(x, y, marker='o', label=label)
    ax.set_title(title)
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    if label:
        ax.legend()
    return ax


import matplotlib.pyplot as plt

def scatter_chart(ax, x, y, title='', xlabel='', ylabel=''):
    ax.scatter(x, y)
    ax.set_title(title)
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    return ax


import matplotlib.pyplot as plt

def histogram(ax, data, bins=10, title='', xlabel=''):
    ax.hist(data, bins=bins, edgecolor='white')
    ax.set_title(title)
    ax.set_xlabel(xlabel)
    ax.set_ylabel('Count')
    return ax


import os
import matplotlib.pyplot as plt

def multi_chart_figure(df, out_path='dashboard.png'):
    fig, axes = plt.subplots(2, 2, figsize=(12, 10))

    # Top-left: bar chart — revenue by product
    totals = df.groupby('product')['revenue'].sum().sort_values(ascending=False)
    bar_chart(axes[0, 0], totals.index.tolist(), totals.values.tolist(),
              'Revenue by Product', 'Product', 'Revenue ($)')

    # Top-right: line chart — cumulative revenue
    df_s = df.sort_values('order_id').reset_index(drop=True)
    line_chart(axes[0, 1], list(range(1, len(df_s) + 1)),
               df_s['revenue'].cumsum().tolist(),
               'Cumulative Revenue', 'Order #', 'Revenue ($)',
               label='cumulative')

    # Bottom-left: scatter — price vs revenue
    scatter_chart(axes[1, 0], df['price'].tolist(), df['revenue'].tolist(),
                  'Price vs Revenue', 'Price ($)', 'Revenue ($)')

    # Bottom-right: histogram — revenue distribution
    histogram(axes[1, 1], df['revenue'].tolist(), bins=8,
              title='Revenue Distribution', xlabel='Revenue ($)')

    fig.suptitle('Sales Dashboard', fontsize=16)
    plt.tight_layout()
    fig.savefig(out_path, bbox_inches='tight', dpi=100)
    plt.close(fig)
    return out_path

## Step 1 — Load Data

In [ ]:
RETAIL_CSV = (
    'order_id,product,category,region,price,quantity\n'
    '1,Widget,Electronics,North,25.0,10\n'
    '2,Gadget,Electronics,South,150.0,3\n'
    '3,Widget,Electronics,South,25.0,5\n'
    '4,Doohickey,Accessories,East,8.0,50\n'
    '5,Gadget,Electronics,East,150.0,7\n'
    '6,Widget,Electronics,East,25.0,4\n'
    '7,Doohickey,Accessories,North,8.0,20\n'
    '8,Gadget,Electronics,North,150.0,2\n'
    '9,Widget,Electronics,West,25.0,6\n'
    '10,Doohickey,Accessories,South,8.0,15\n'
    '11,Thingamajig,Accessories,North,200.0,1\n'
    '12,Thingamajig,Accessories,East,200.0,4'
)
df = pd.read_csv(io.StringIO(RETAIL_CSV))
df['revenue'] = df['price'] * df['quantity']
print(f'Shape: {df.shape}')
print(df[['product', 'price', 'quantity', 'revenue']].to_string(index=False))

assert df.shape == (12, 7)
assert 'revenue' in df.columns

## Step 2 — Individual Charts

In [ ]:
# Bar chart — revenue by product
_fig, _ax = plt.subplots(figsize=(7, 4))
totals = df.groupby('product')['revenue'].sum().sort_values(ascending=False)
bar_chart(_ax, totals.index.tolist(), totals.values.tolist(),
          'Revenue by Product', 'Product', 'Revenue ($)')
_fig.savefig('/tmp/day039_bar.png', bbox_inches='tight', dpi=72)
plt.close(_fig)
print('bar_chart: saved')

# Line chart — cumulative revenue
_fig, _ax = plt.subplots(figsize=(7, 4))
df_s = df.sort_values('order_id').reset_index(drop=True)
line_chart(_ax, list(range(1, len(df_s) + 1)),
           df_s['revenue'].cumsum().tolist(),
           'Cumulative Revenue', 'Order #', 'Revenue ($)', label='cumulative')
_fig.savefig('/tmp/day039_line.png', bbox_inches='tight', dpi=72)
plt.close(_fig)
print('line_chart: saved')

# Scatter chart — price vs revenue
_fig, _ax = plt.subplots(figsize=(7, 4))
scatter_chart(_ax, df['price'].tolist(), df['revenue'].tolist(),
              'Price vs Revenue', 'Price ($)', 'Revenue ($)')
_fig.savefig('/tmp/day039_scatter.png', bbox_inches='tight', dpi=72)
plt.close(_fig)
print('scatter_chart: saved')

# Histogram — revenue distribution
_fig, _ax = plt.subplots(figsize=(7, 4))
histogram(_ax, df['revenue'].tolist(), bins=8,
          title='Revenue Distribution', xlabel='Revenue ($)')
_fig.savefig('/tmp/day039_hist.png', bbox_inches='tight', dpi=72)
plt.close(_fig)
print('histogram: saved')

assert len(plt.get_fignums()) == 0, 'all figures should be closed'

## Step 3 — Dashboard (2×2)

In [ ]:
out = multi_chart_figure(df, 'chart_dashboard.png')
print(f'Dashboard saved to: {out}')

assert os.path.exists('chart_dashboard.png')
sz = os.path.getsize('chart_dashboard.png')
assert sz > 10000, f'file too small: {sz} bytes'
with open('chart_dashboard.png', 'rb') as f:
    assert f.read(4) == b'\x89PNG'
assert len(plt.get_fignums()) == 0
print(f'Dashboard verified: {sz:,} bytes, valid PNG, no open figures')

## Step 4 — Individual Chart Verification

In [ ]:
# Verify each chart type's properties independently
_fig, _ax = plt.subplots()
bar_chart(_ax, ['A', 'B', 'C'], [10, 20, 15], 'Test', 'X', 'Y')
assert len(_ax.patches) == 3
assert _ax.get_title() == 'Test'
plt.close(_fig)
print('bar_chart: 3 bars, title correct')

_fig, _ax = plt.subplots()
line_chart(_ax, [1, 2, 3], [10, 20, 15], label='s')
assert len(_ax.get_lines()) >= 1
assert _ax.get_legend() is not None
plt.close(_fig)
print('line_chart: line + legend correct')

_fig, _ax = plt.subplots()
scatter_chart(_ax, [1, 2, 3], [4, 5, 6])
assert len(_ax.collections) >= 1
assert _ax.collections[0].get_offsets().shape[0] == 3
plt.close(_fig)
print('scatter_chart: 3 points correct')

_fig, _ax = plt.subplots()
histogram(_ax, [1, 2, 3, 4, 5, 6, 7, 8], bins=4)
assert len(_ax.patches) > 0
assert _ax.get_ylabel() == 'Count'
plt.close(_fig)
print('histogram: bars + Count label correct')

print('\nAll charts verified!')